## Import

In [30]:
import random
import pandas as pd
import numpy as np
import os
import re
import glob
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor


import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2
import torchvision.models as models

from tqdm import tqdm

import warnings
warnings.filterwarnings(action='ignore') 

In [31]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

## Hyperparameter Setting

In [32]:
CFG = {
    'IMG_SIZE':224,
    'EPOCHS':100,
    'LEARNING_RATE':1e-3,
    'BATCH_SIZE':32,
    'SEED':777
}

## Fixed RandomSeed

In [33]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG['SEED'])

## Data Pre-processing

In [34]:
df = pd.read_csv('./train.csv')

In [35]:
train_len = int(len(df) * 0.8)
train_df = df.iloc[:train_len]

remaining_df = df.iloc[train_len:]
val_len = int(len(remaining_df) * 0.5)

val_df = remaining_df.iloc[:val_len]
test_df = remaining_df.iloc[val_len:]

In [36]:
train_label_vec = train_df.iloc[:, 2:].values.astype(np.float32)
val_label_vec = val_df.iloc[:, 2:].values.astype(np.float32)
test_label_vec = test_df.iloc[:, 2:].values.astype(np.float32)

In [37]:
CFG['label_size'] = train_label_vec.shape[1]

## CustomDataset

In [38]:
class CustomDataset(Dataset):
    def __init__(self, img_path_list, label_list, transforms=None):
        self.img_path_list = img_path_list
        self.label_list = label_list
        self.transforms = transforms
        
    def __getitem__(self, index):
        img_path = self.img_path_list[index]
        
        image = cv2.imread(img_path)
        
        if self.transforms is not None:
            image = self.transforms(image=image)['image']
        
        if self.label_list is not None:
            label = self.label_list[index]
            return image, label
        else:
            return image
        
    def __len__(self):
        return len(self.img_path_list)

In [39]:
train_transform = A.Compose([
                            A.Resize(CFG['IMG_SIZE'],CFG['IMG_SIZE']),
                            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, always_apply=False, p=1.0),
                            ToTensorV2()
                            ])

test_transform = A.Compose([
                            A.Resize(CFG['IMG_SIZE'],CFG['IMG_SIZE']),
                            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, always_apply=False, p=1.0),
                            ToTensorV2()
                            ])

In [40]:
train_dataset = CustomDataset(train_df['path'].values, train_label_vec, train_transform)
train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True, num_workers=0)

val_dataset = CustomDataset(val_df['path'].values, val_label_vec, test_transform)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

test_dataset = CustomDataset(test_df['path'].values, None, test_transform)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

## Model Define

In [41]:
class Efficientnet(nn.Module):
    def __init__(self, gene_size=CFG['label_size'], dropout_rate=0.5):
        super(Efficientnet, self).__init__()
        self.backbone = models.efficientnet_b0(pretrained=True)
        self.backbone.classifier = nn.Identity() 
        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(1280, gene_size) 
        )
        
    def forward(self, x):
        x = self.backbone(x)  
        x = self.regressor(x) 
        return x

In [42]:
class ResNetModel(nn.Module):
    def __init__(self, gene_size=CFG['label_size'], dropout_rate=0.5):
        super(ResNetModel, self).__init__()
        self.backbone = models.resnet50(pretrained=True)
        self.backbone.fc = nn.Identity() 
        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Dropout(p=dropout_rate), 
            nn.Linear(2048, gene_size) 
        )
        
    def forward(self, x):
        x = self.backbone(x)
        x = self.regressor(x)
        return x


## Train

In [43]:
def train(model, optimizer, train_loader, val_loader, scheduler, device):
    model.to(device)
    criterion = nn.MSELoss().to(device)
    
    best_loss = float('inf')
    best_model = None
    
    for epoch in range(1, CFG['EPOCHS']+1):
        model.train()
        train_loss = []
        for imgs, labels in tqdm(iter(train_loader)):
            imgs = imgs.float().to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            output = model(imgs)
            loss = criterion(output, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss.append(loss.item())
                    
        _val_loss = validation(model, criterion, val_loader, device)
        _train_loss = np.mean(train_loss)
        print(f'Epoch [{epoch}], Train Loss : [{_train_loss:.5f}] Val Loss : [{_val_loss:.5f}]')
       
        if scheduler is not None:
            scheduler.step(_val_loss)
            
        if best_loss > _val_loss:
            best_loss = _val_loss
            best_model = model
    
    return best_model

In [44]:
def validation(model, criterion, val_loader, device):
    model.eval()
    val_loss = []

    with torch.no_grad():
        for imgs, labels in tqdm(iter(val_loader)):
            imgs = imgs.float().to(device)
            labels = labels.to(device)
            
            pred = model(imgs)
            
            loss = criterion(pred, labels)
            
            val_loss.append(loss.item())
        
        _val_loss = np.mean(val_loss)
    
    return _val_loss

In [45]:
def test(model, criterion, test_loader, device):
    model.eval()
    test_loss = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(iter(test_loader)):
            labels = labels.to(device)

            pred = model(imgs)

            loss = criterion(pred, labels)
            test_loss.append(loss.item())

            all_preds.append(pred.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
        _test_loss = np.mean(test_loss)

    print(f'Test Loss : [{_test_loss:.5f}]')
    return _test_loss, all_preds, all_labels

## Run!!

In [46]:
resnet_model = ResNetModel(gene_size=CFG['label_size'], dropout_rate=0.5)
optimizer_resnet = torch.optim.Adam(params=resnet_model.parameters(), lr=CFG["LEARNING_RATE"])
scheduler_resnet = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_resnet, mode='min', factor=0.5, patience=2, threshold_mode='abs', min_lr=1e-8, verbose=True)
trained_resnet_model = train(resnet_model, optimizer_resnet, train_loader, val_loader, scheduler_resnet, device)

efficient_models = []
for i, lr in enumerate([1.0, 0.8, 1.2, 0.9, 1.1]):
    model = Efficientnet(gene_size=CFG['label_size'], dropout_rate=0.5)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=CFG["LEARNING_RATE"] * lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, threshold_mode='abs', min_lr=1e-8, verbose=True)
    trained_model = train(model, optimizer, train_loader, val_loader, scheduler, device)
    efficient_models.append(trained_model)

trained_model1, trained_model2, trained_model3, trained_model4, trained_model5 = efficient_models

print("All models trained successfully.")

100%|██████████| 22/22 [00:03<00:00,  6.22it/s]


Epoch [1], Train Loss : [0.05529] Val Loss : [0.04708]


100%|██████████| 22/22 [00:03<00:00,  6.22it/s]


Epoch [2], Train Loss : [0.04838] Val Loss : [0.04684]


100%|██████████| 22/22 [00:03<00:00,  6.20it/s]


...

Epoch [98], Train Loss : [0.04410] Val Loss : [0.04574]


100%|██████████| 22/22 [00:02<00:00,  9.71it/s]


Epoch [99], Train Loss : [0.04410] Val Loss : [0.04573]


100%|██████████| 22/22 [00:02<00:00,  9.84it/s]

Epoch [100], Train Loss : [0.04409] Val Loss : [0.04573]
All models trained successfully.


## Inference

In [47]:
test = pd.read_csv('./test.csv')

In [48]:
test_dataset = CustomDataset(test['path'].values, None, test_transform)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

In [49]:
def inference(model, test_loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for imgs in tqdm(test_loader):
            imgs = imgs.to(device).float()
            pred = model(imgs)
            
            preds.append(pred.detach().cpu())
    
    preds = torch.cat(preds).numpy()

    return preds

In [50]:
def inference(model, data_loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for data in tqdm(data_loader):
            imgs = data[0].to(device).float()  # imgs만 추출하고 to(device) 사용
            
            # 4D 텐서인지 확인 후, 아니면 차원을 추가
            if imgs.dim() == 3:  
                imgs = imgs.unsqueeze(0)  # 배치 차원 추가
            
            pred = model(imgs)
            preds.append(pred.detach().cpu())
    
    preds = torch.cat(preds).numpy()
    return preds


## Submission

In [51]:
# 1. Validation predictions
val_preds1 = inference(trained_model1, val_loader, device)
val_preds2 = inference(trained_model2, val_loader, device)
val_preds3 = inference(trained_model3, val_loader, device)
val_preds4 = inference(trained_model4, val_loader, device)
val_preds5 = inference(trained_model5, val_loader, device)
val_preds_resnet = inference(trained_resnet_model, val_loader, device)

# 2. Weighted ensemble for validation
val_weighted_ensemble_preds = (val_preds1 * weights[0] + val_preds2 * weights[1] + 
                               val_preds3 * weights[2] + val_preds4 * weights[3] + 
                               val_preds5 * weights[4])

# 3. Stacking data preparation for validation
stacked_preds_val = np.column_stack((val_preds1, val_preds2, val_preds3, val_preds4, 
                                     val_preds5, val_weighted_ensemble_preds, val_preds_resnet))

# 4. Meta model training using validation set predictions
meta_model = LinearRegression()
meta_model.fit(stacked_preds_val, val_label_vec)  # Use validation labels here

# 5. Test predictions
test_preds1 = inference(trained_model1, test_loader, device)
test_preds2 = inference(trained_model2, test_loader, device)
test_preds3 = inference(trained_model3, test_loader, device)
test_preds4 = inference(trained_model4, test_loader, device)
test_preds5 = inference(trained_model5, test_loader, device)
test_preds_resnet = inference(trained_resnet_model, test_loader, device)

# 6. Weighted ensemble for test
test_weighted_ensemble_preds = (test_preds1 * weights[0] + test_preds2 * weights[1] + 
                                test_preds3 * weights[2] + test_preds4 * weights[3] + 
                                test_preds5 * weights[4])

# 7. Stacking data preparation for test
stacked_preds_test = np.column_stack((test_preds1, test_preds2, test_preds3, test_preds4, 
                                      test_preds5, test_weighted_ensemble_preds, test_preds_resnet))

# 8. Meta model predictions on test set
stacked_ensemble_preds_test = meta_model.predict(stacked_preds_test)

# 9. Save the final predictions
# 전체 테스트 데이터에 대한 예측 수행
stacked_ensemble_preds_test = meta_model.predict(stacked_preds_test)  # stacked_preds_test는 전체 테스트 데이터에 대한 예측

# 예측값의 길이 확인
print(f"Length of stacked_ensemble_preds_test: {len(stacked_ensemble_preds_test)}")  # 예상한 길이 확인

# 제출 파일 생성
submit = pd.read_csv('./sample_submission.csv')

# 예측값의 길이와 submit 데이터프레임의 길이 확인
print(f"Length of submit DataFrame: {len(submit)}")  # 2277여야 함

# 예측값을 submit 데이터프레임에 저장
if len(stacked_ensemble_preds_test) == len(submit) - 1:  # 1 열 제외
    submit.iloc[:, 1:] = stacked_ensemble_preds_test.astype(np.float32)
else:
    print("The length of the predictions does not match the expected length.")

# 제출 파일 저장
submit.to_csv('./stacked_weighted_ensemble_resnet_efficientnet_submit.csv', index=False)

print("Final predictions saved to stacked_weighted_ensemble_resnet_efficientnet_submit.csv")


100%|██████████| 72/72 [00:05<00:00, 12.56it/s]


Length of stacked_ensemble_preds_test: 72
Length of submit DataFrame: 2277
The length of the predictions does not match the expected length.
Final predictions saved to stacked_weighted_ensemble_resnet_efficientnet_submit.csv
